<a href="https://colab.research.google.com/github/askSayyam/CrisisLens/blob/main/BM25_%26_hybrid_retrieval_function_(BM25%2BFAISS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# phase 3-

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/CrisisLens/'
print("Drive mounted ✓")
print(f"Path: {DRIVE}")

In [ ]:
import pandas as pd
import numpy as np

import os
!pip install faiss-cpu > /dev/null
import faiss
from sentence_transformers import SentenceTransformer

# ── Load all 5 files ──
unified    = pd.read_csv(DRIVE + 'unified_corpus.csv')
queries_df = pd.read_csv(DRIVE + 'queries.csv')
qrels_df   = pd.read_csv(DRIVE + 'qrels.csv')
posts_df   = pd.read_csv(DRIVE + 'posts_clean.csv')
pairs_df   = pd.read_csv(DRIVE + 'pairs_dev_crosslingual.csv')

print(f"unified_corpus      : {len(unified):,} rows")
print(f"queries             : {len(queries_df):,} rows")
print(f"qrels               : {len(qrels_df):,} rows")
print(f"posts_clean         : {len(posts_df):,} rows")
print(f"pairs_crosslingual  : {len(pairs_df):,} rows")
print("All 5 files loaded ✓")

# ── Load LaBSE from Drive ──
model = SentenceTransformer(DRIVE + 'labse_model/')
print(f"LaBSE loaded ✓")

# ── Load embeddings ──
embs = np.load(DRIVE + 'corpus_embeddings.npy').astype('float32')
uids = np.load(DRIVE + 'corpus_uids.npy', allow_pickle=True)
print(f"Embeddings loaded ✓ shape : {embs.shape}")
print(f"UIDs loaded       ✓ count : {len(uids):,}")

# ── Load FAISS index ──
index = faiss.read_index(DRIVE + 'corpus.faiss')
print(f"FAISS index loaded ✓ vectors: {index.ntotal:,}")

print("\nAll loaded. Ready ✓")

In [ ]:
#BM25
!pip install rank_bm25 > /dev/null
from rank_bm25 import BM25Okapi
import pickle
import re
from tqdm import tqdm

# ── Tokenizer ──
def tokenize(text):
    if not isinstance(text, str): return []
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text.split()

# ── Tokenize all passages ──
print(f"Tokenizing {len(unified):,} passages...")
texts     = unified['text'].fillna('').tolist()
tokenized = [tokenize(t) for t in tqdm(texts)]
print("Tokenization done ✓")

# ── Build BM25 ──
print("\nBuilding BM25 index...")
bm25 = BM25Okapi(tokenized)
print("BM25 built ✓")

# ── Save to Drive ──
with open(DRIVE + 'bm25.pkl', 'wb') as f:
    pickle.dump(bm25, f)
print("bm25.pkl saved to Drive ✓")

# ── Verify ──
with open(DRIVE + 'bm25.pkl', 'rb') as f:
    bm25_check = pickle.load(f)

# ── Quick test ──
test_tokens = tokenize("flood evacuation emergency shelter")
scores      = bm25_check.get_scores(test_tokens)
top3        = scores.argsort()[::-1][:3]

print(f"\nTop 3 BM25 results for 'flood evacuation emergency shelter':")
for rank, idx in enumerate(top3, 1):
    print(f"  {rank}. [{unified.iloc[idx]['uid']}] score={scores[idx]:.3f}")
    print(f"     {unified.iloc[idx]['text'][:80]}...")

print(f"\nBM25 index complete ✓")
print(f"Never runs again — loads from Drive next time")

In [ ]:
# the hybrid retrieval function (BM25 + FAISS combined):
import numpy as np
import re

# ── Tokenizer (same as before) ──
def tokenize(text):
    if not isinstance(text, str): return []
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text.split()

# ── Normalize scores to 0-1 ──
def minmax_norm(arr):
    mn, mx = arr.min(), arr.max()
    if mx == mn: return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

# ── Hybrid retrieval function ──
def hybrid_retrieve(query_text: str,
                    top_k: int = 5,
                    alpha: float = 0.6) -> list:
    """
    alpha = 0.6 means 60% dense (LaBSE) + 40% sparse (BM25)
    Returns top_k results as list of dicts
    """
    # ── Dense retrieval (FAISS) ──
    q_vec    = model.encode([query_text],
                             normalize_embeddings=True).astype('float32')
    d_scores, d_indices = index.search(q_vec, 200)
    dense_map = {uids[i]: s
                 for i, s in zip(d_indices[0], d_scores[0]) if i >= 0}

    # ── Sparse retrieval (BM25) ──
    tokens   = tokenize(query_text)
    b_scores = bm25.get_scores(tokens)
    top200_b = b_scores.argsort()[::-1][:200]
    bm25_map = {uids[i]: b_scores[i] for i in top200_b}

    # ── Merge candidate pool ──
    all_uids = list(set(dense_map) | set(bm25_map))
    d_arr    = np.array([dense_map.get(u, 0.0) for u in all_uids])
    b_arr    = np.array([bm25_map.get(u, 0.0)  for u in all_uids])

    # ── Normalize both ──
    d_norm = minmax_norm(d_arr)
    b_norm = minmax_norm(b_arr)

    # ── Combine ──
    combined = alpha * d_norm + (1 - alpha) * b_norm

    # ── Sort and return top_k ──
    sorted_idx = combined.argsort()[::-1][:top_k]

    results = []
    for idx in sorted_idx:
        uid = all_uids[idx]
        row = unified[unified['uid'] == uid].iloc[0]
        results.append({
            'uid'     : uid,
            'text'    : row['text'],
            'lang'    : row['lang'],
            'source'  : row['source'],
            'score'   : round(float(combined[idx]), 4)
        })
    return results

# ── Test in 4 languages ──
test_queries = [
    ("eng", "flood evacuation emergency shelter"),
    ("urd", "سیلاب سے بچاؤ"),
    ("ara", "إخلاء الفيضانات"),
    ("fra", "évacuation inondations"),
]

print("Testing hybrid retrieval across languages:")
print("=" * 60)
for lang, q in test_queries:
    print(f"\n[{lang}] '{q}'")
    results = hybrid_retrieve(q, top_k=3)
    for r in results:
        print(f"  {r['score']} | [{r['lang']}] {r['text'][:70]}...")